In [1]:
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langchain-text-splitters==1.1.2" "python-dotenv==1.2.3"


Note: you may need to restart the kernel to use updated packages.


<h3>개념 요약</h3>
<p>검색 증강 생성(RAG)은 문서를 미리 색인하는 흐름과 질문마다 관련 문서를 찾는 흐름으로 나뉩니다. 이 절에서는 검색 문맥이 없을 때 생성 모델이 사내 규정을 답할 수 없는 이유부터 확인합니다.</p>

In [2]:
import os

from dotenv import find_dotenv, load_dotenv

env_file = find_dotenv(usecwd=True)
assert env_file and load_dotenv(env_file), '프로젝트 루트 .env를 찾지 못했습니다.'
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY가 없습니다.'
print('환경 준비 완료')


환경 준비 완료


<h3>예제 실행</h3>
<p>가상 기관의 규정을 검색 문맥 없이 실제 생성 모델에 질문합니다.</p>

In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model('openai:gpt-5.4-mini', temperature=0)
question = '가상 기관 파란도서관의 정기 휴관일은 언제인가요?'
reply = model.invoke(question)
print(reply.text)


파란도서관이 **가상 기관**이라면, 정기 휴관일은 제가 임의로 확정할 수 없어요.  
보통은 설정에 따라 다르므로, 아래 중 하나를 정하면 됩니다:

- **매주 월요일 휴관**
- **매주 일요일 휴관**
- **공휴일 및 매월 첫째 주 월요일 휴관**
- **운영 규정에 별도 명시**

원하시면 제가 **파란도서관의 운영 규정** 형태로 정기 휴관일을 자연스럽게 만들어드릴게요.


<h3>변형 실행</h3>
<p><code>question</code>을 다른 가상 기관의 내부 규정 질문으로 바꾸어 한 번 더 실행합니다. 모델이 공개 지식과 사내 문서를 구분해 답하는지 확인합니다.</p>

In [4]:
question = '가상 회사 해솔상사의 경조 휴가는 며칠인가요?'
changed = model.invoke(question)
print(changed.text)


가상 회사 **해솔상사**의 **경조 휴가 일수**는, 제가 현재 알고 있는 정보만으로는 확인할 수 없습니다.  
해솔상사의 **사내 규정집, 복리후생 안내, 인사 규정**에 명시된 내용을 봐야 정확히 알 수 있어요.

원하시면 제가 대신 도와드릴 수 있는 방법은:
1. **경조 휴가 규정 문구를 보내주시면** 며칠인지 바로 정리해드리기  
2. **일반적인 한국 회사의 경조휴가 기준**을 참고용으로 알려드리기  
3. 해솔상사용으로 **가상의 사내 규정 예시**를 만들어드리기

원하시는 방향을 말씀해 주세요.


<h3>실행 검증</h3>
<p>앞 셀의 결과가 이 절에서 확인할 조건을 만족하는지 단언문으로 검사합니다.</p>

In [5]:
assert reply.text and changed.text
assert '[문서 p.' not in changed.text
print('검증 통과: 생성 호출은 성공했고 RAG 문맥은 아직 없습니다.')


검증 통과: 생성 호출은 성공했고 RAG 문맥은 아직 없습니다.


<h3>정리</h3>
<p>생성 모델 호출만으로는 사내 문서의 답과 출처를 보장할 수 없습니다. 다음 절부터 문서를 나누고 검색 가능한 형태로 준비합니다.</p>

<p><strong>원인</strong> — 문서 문맥을 제공하지 않은 응답에서 사내 규정의 출처를 검사하면 실패합니다.</p>

In [6]:
before_reply = changed
print(before_reply.text)
try:
    assert '[해솔상사 규정 p.1]' in before_reply.text
except AssertionError:
    print('예상한 실패: 제공된 근거와 출처가 없습니다.')


가상 회사 **해솔상사**의 **경조 휴가 일수**는, 제가 현재 알고 있는 정보만으로는 확인할 수 없습니다.  
해솔상사의 **사내 규정집, 복리후생 안내, 인사 규정**에 명시된 내용을 봐야 정확히 알 수 있어요.

원하시면 제가 대신 도와드릴 수 있는 방법은:
1. **경조 휴가 규정 문구를 보내주시면** 며칠인지 바로 정리해드리기  
2. **일반적인 한국 회사의 경조휴가 기준**을 참고용으로 알려드리기  
3. 해솔상사용으로 **가상의 사내 규정 예시**를 만들어드리기

원하시는 방향을 말씀해 주세요.
예상한 실패: 제공된 근거와 출처가 없습니다.


<p><strong>수정</strong> — 질문과 함께 짧은 규정 문장과 출처 태그를 제공합니다.</p>

In [7]:
evidence = '[해솔상사 규정 p.1] 본인 결혼 경조 휴가는 5일입니다.'
after_prompt = f'다음 근거만 사용해 답하고 출처를 붙이세요.\n{evidence}\n질문: {question}'
after_reply = model.invoke(after_prompt)
print(after_reply.text)
assert '5일' in after_reply.text
assert '[해솔상사 규정 p.1]' in after_reply.text


가상 회사 해솔상사의 본인 결혼 경조 휴가는 **5일**입니다.  
출처: **[해솔상사 규정 p.1] 본인 결혼 경조 휴가는 5일입니다.**
